# Minimal CPU training-preprocessing step

This notebook validates one raw mmCIF input and runs only the Protenix training **preprocessor**. It does not rewrite the CIF, inject components, use a parity adapter, or run model forward/backward.

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import hashlib
from pathlib import Path
import torch

from ofoldx.data.molecular import PolymerSequence
from ofoldx.io import read_structure
from ofoldx.io.mmcif import parse_mmcif
from ofoldx.models.protenix import ProtenixProcessor, ProtenixProcessorConfig

SOURCE = Path('/data/shared/datasets/protenix/mmcif/6hlv.cif')
STANDARD_AA = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'}
ALLOWED_ELEMENTS = {'C', 'N', 'O', 'S'}

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_sha_before = sha256(SOURCE)
print('device policy: CPU-only; CUDA_VISIBLE_DEVICES=', repr(os.environ['CUDA_VISIBLE_DEVICES']))
print('source:', SOURCE)
print('source sha256 before:', source_sha_before)

device policy: CPU-only; CUDA_VISIBLE_DEVICES= ''
source: /data/shared/datasets/protenix/mmcif/6hlv.cif
source sha256 before: 0706ad8520a78e59b27e3551f9cd26b82dd66319e93ef93913269b3fe9d2607d


## Validate the untouched model-1 source

The raw parser is used only for selection checks. The structure passed to the processor is then obtained directly with `read_structure(SOURCE, model=1)`.

In [2]:
parsed = parse_mmcif(SOURCE, model=1, assembly_id=None)
groups = {row.group for row in parsed.rows}
source_elements = {row.element.upper() for row in parsed.rows}
assert groups == {'ATOM'}, groups
assert source_elements <= ALLOWED_ELEMENTS, source_elements - ALLOWED_ELEMENTS

structure = read_structure(SOURCE, model=1)
entities = structure.system.entities
assert all(isinstance(entity, PolymerSequence) and entity.polymer_type == 'protein' for entity in entities.values())
protein_instances = sum(isinstance(entities[eid], PolymerSequence) and entities[eid].polymer_type == 'protein' for eid in structure.system.instances.values())
component_ids = {component.component_id for entity in entities.values() for component in entity.components}
observed_elements = {structure.system.atom(site).element.upper() for site in structure.sites}
assert protein_instances >= 2, protein_instances
assert component_ids <= STANDARD_AA, component_ids - STANDARD_AA
assert observed_elements <= ALLOWED_ELEMENTS, observed_elements - ALLOWED_ELEMENTS

print('model-1 groups:', sorted(groups))
print('model-1 source elements:', sorted(source_elements))
print('observed elements:', sorted(observed_elements))
print('protein instances:', protein_instances)
print('standard component IDs:', sorted(component_ids))
print('observed/resolved sites:', len(structure.sites), '/', int(structure.coordinate_mask.sum()))

model-1 groups: ['ATOM']
model-1 source elements: ['C', 'N', 'O', 'S']
observed elements: ['C', 'N', 'O', 'S']
protein instances: 2
standard component IDs: ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL']
observed/resolved sites: 1383 / 1383


## Run query-only training preprocessing

`msa=None` creates a query-only MSA (depth 1). Reference-position augmentation is disabled and every produced tensor is required to remain on CPU.

In [3]:
processor = ProtenixProcessor(ProtenixProcessorConfig(augment_reference_positions=False), device='cpu')
processed, target = processor.process_training(structure, msa=None)

token_count = len(processed.token_sites)
atom_count = len(processed.atom_sites)
assert processed.msa_tokens.shape == (1, 1, token_count)
assert processed.has_deletion.shape == (1, 1, token_count)
assert processed.deletion_value.shape == (1, 1, token_count)
assert target.coordinates.shape == (1, len(structure.sites), 3)
assert target.coordinate_mask.shape == (1, len(structure.sites))
assert target.atom_index.shape == (1, atom_count)

key_floats = {
    'atom_features': processed.atom_features,
    'reference_positions': processed.reference_positions,
    'token_features': processed.token_features,
    'deletion_value': processed.deletion_value,
}
for name, tensor in key_floats.items():
    assert tensor.device.type == 'cpu'
    assert torch.isfinite(tensor).all(), name
resolved = target.coordinate_mask[0]
mapped = target.atom_index[0] >= 0
assert torch.isfinite(target.coordinates[0, resolved]).all()
assert int(mapped.sum()) == len(structure.sites)

print('query-only MSA depth:', processed.msa_tokens.shape[1])
print('token shapes:', {'token_features': tuple(processed.token_features.shape), 'msa_tokens': tuple(processed.msa_tokens.shape)})
print('atom shapes:', {'atom_features': tuple(processed.atom_features.shape), 'reference_positions': tuple(processed.reference_positions.shape)})
print('target shapes:', {'coordinates': tuple(target.coordinates.shape), 'coordinate_mask': tuple(target.coordinate_mask.shape), 'atom_index': tuple(target.atom_index.shape)})
print('resolved target atoms:', int(resolved.sum()))
print('mapped target atoms:', int(mapped.sum()))
print('key float tensors finite:', {name: bool(torch.isfinite(tensor).all()) for name, tensor in key_floats.items()})

query-only MSA depth: 1
token shapes: {'token_features': (1, 224, 449), 'msa_tokens': (1, 1, 224)}
atom shapes: {'atom_features': (1, 1827, 385), 'reference_positions': (1, 1827, 3)}
target shapes: {'coordinates': (1, 1383, 3), 'coordinate_mask': (1, 1383), 'atom_index': (1, 1827)}
resolved target atoms: 1383
mapped target atoms: 1383
key float tensors finite: {'atom_features': True, 'reference_positions': True, 'token_features': True, 'deletion_value': True}


## Confirm source integrity

Re-hash the original file after preprocessing. No model is instantiated and no forward or backward call occurs.

In [4]:
source_sha_after = sha256(SOURCE)
assert source_sha_after == source_sha_before
print('source sha256 after:', source_sha_after)
print('source hash unchanged:', source_sha_after == source_sha_before)
print('model forward/backward performed:', False)
print('OFOLDX_TRAIN_STEP_NOTEBOOK_OK_6HLV')

source sha256 after: 0706ad8520a78e59b27e3551f9cd26b82dd66319e93ef93913269b3fe9d2607d
source hash unchanged: True
model forward/backward performed: False
OFOLDX_TRAIN_STEP_NOTEBOOK_OK_6HLV
